<a href="https://colab.research.google.com/github/alyssaplayer/BU_OMDS_APlayerRepo/blob/main/Capstone/DX799_Capstone_USRecords_(ECommerce).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Project — Summer Semester OMDS 2026

## United States E-Commerce Records (2020)

This notebook analyses the US E-Commerce Records (2020) dataset sourced from Kaggle (Ahmad, 2020). The dataset contains approximately 3,000 rows covering sales transactions across product categories, US regions, customer segments, and order dates. The central analytical goal is to **predict Profit** at the order level and to **classify orders as high- or low-profit** — providing actionable insights for e-commerce retailers seeking to optimise regional inventory, refine marketing strategy, and anticipate demand shifts stemming from the COVID-19 pandemic's impact on US online purchasing behaviour.

**Dataset features:** Order ID, Order Date, Ship Mode, Customer ID/Name, Segment, Country, City, State, Postal Code, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit.

# Load Required Libraries

In [ ]:
#pip install pandas matplotlib seaborn numpy kagglehub scikit-learn


In [ ]:
import kagglehub
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

# Load Data

In [ ]:
# U.S. E-Commerce Records (2020)
# Source: https://www.kaggle.com/datasets/ammaraahmad/us-ecommerce-record-2020

path = kagglehub.dataset_download("ammaraahmad/us-ecommerce-record-2020")
csv_files = glob.glob(os.path.join(path, '*.csv'))

if csv_files:
    df_raw = pd.read_csv(csv_files[0], encoding='latin1')
    print(f"Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
    display(df_raw.head())
else:
    print("No CSV files found.")

# Exploratory Data Analysis (EDA)

We begin with a high-level overview of the data: shape, data types, missing values, and summary statistics — mirroring the EDA approach from the Olist notebook.

In [ ]:
# Shape, dtypes, and null counts
print("Shape:", df_raw.shape)
print("\nColumn dtypes:")
print(df_raw.dtypes)
print("\nMissing values per column:")
print(df_raw.isnull().sum())

In [ ]:
# Summary statistics for numeric features
display(df_raw.describe())

In [ ]:
# Categorical value counts
for col in ['Category', 'Sub-Category', 'Segment', 'Region', 'Ship Mode']:
    print(f"\n{col}:")
    print(df_raw[col].value_counts())

## EDA Visualisations

In [ ]:
# --- 1. Distribution of Profit (raw) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_raw['Profit'], bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_title('Profit Distribution (Raw)')
axes[0].set_xlabel('Profit ($)')
axes[0].set_ylabel('Count')

axes[1].boxplot(df_raw['Profit'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Profit Boxplot (Raw)')
axes[1].set_ylabel('Profit ($)')

plt.tight_layout()
plt.show()
print(f"Profit skewness: {df_raw['Profit'].skew():.3f}")

In [ ]:
# --- 2. Correlation Matrix of Numeric Features ---
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
corr_matrix = df_raw[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, annot_kws={'size': 11})
plt.title('Correlation Matrix — Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# --- 3. Average Profit by Category and Region (Heatmap) ---
pivot = df_raw.pivot_table(values='Profit', index='Region', columns='Category', aggfunc='mean')

plt.figure(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', linewidths=0.5,
            annot_kws={'size': 10})
plt.title('Average Profit by Region and Product Category')
plt.tight_layout()
plt.show()

In [ ]:
# --- 4. Monthly Profit Trend by Category ---
df_plot = df_raw.copy()
df_plot['Order Date'] = pd.to_datetime(df_plot['Order Date'], format='%d-%m-%y')
df_plot['YearMonth'] = df_plot['Order Date'].dt.to_period('M').dt.to_timestamp()

monthly_cat = df_plot.groupby(['YearMonth', 'Category'])['Profit'].sum().reset_index()

plt.figure(figsize=(14, 5))
for cat, grp in monthly_cat.groupby('Category'):
    plt.plot(grp['YearMonth'], grp['Profit'], marker='o', markersize=4, label=cat)
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Monthly Total Profit by Product Category (2020)')
plt.xlabel('Month')
plt.ylabel('Total Profit ($)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Feature Engineering

We engineer features from the raw dataset to support supervised and unsupervised modelling. This includes:
- Datetime decomposition (year, month, day-of-week, weekend flag)
- Clipping the top and bottom 1% of Profit to reduce outlier skew (as identified in EDA)
- One-hot encoding of categorical predictors (Category, Sub-Category, Region, Segment, Ship Mode)
- A binary target variable `High_Profit` for classification models (W4, W5, W6)

In [ ]:
df = df_raw.copy()

# --- Fix data types ---
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%y')
df['Postal Code'] = df['Postal Code'].astype(str)

# --- Date signals ---
df['Order_Year']      = df['Order Date'].dt.year
df['Order_Month']     = df['Order Date'].dt.month
df['Order_DayOfWeek'] = df['Order Date'].dt.dayofweek   # Mon=0, Sun=6
df['Is_Weekend']      = (df['Order_DayOfWeek'] >= 5).astype(int)

# --- Clip Profit outliers (top & bottom 1%) ---
lower = df['Profit'].quantile(0.01)
upper = df['Profit'].quantile(0.99)
df['Profit_clipped'] = df['Profit'].clip(lower, upper)
print(f"Profit clipped to [{lower:.2f}, {upper:.2f}]")

# --- Binary target: 1 = above-median profit, 0 = below ---
profit_median = df['Profit_clipped'].median()
df['High_Profit'] = (df['Profit_clipped'] > profit_median).astype(int)
print(f"Median profit (clipped): ${profit_median:.2f}")
print(f"High_Profit class balance:\n{df['High_Profit'].value_counts()}")

In [ ]:
# --- One-hot encode categorical features ---
cat_features = ['Category', 'Sub-Category', 'Segment', 'Region', 'Ship Mode']
df_encoded = pd.get_dummies(df, columns=cat_features, drop_first=True)

# Regression feature set (numeric + encoded categoricals + date signals)
feature_cols = (
    ['Sales', 'Quantity', 'Discount',
     'Order_Month', 'Order_DayOfWeek', 'Is_Weekend']
    + [c for c in df_encoded.columns
       if any(c.startswith(p + '_') for p in cat_features)]
)

# Remove any booleans -> int for sklearn compatibility
df_encoded[feature_cols] = df_encoded[feature_cols].astype(float)

print(f"Feature set: {len(feature_cols)} columns")
print(feature_cols)

In [ ]:
# --- Visualise Profit before vs after clipping ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['Profit'], bins=60, color='tomato', edgecolor='white', linewidth=0.4)
axes[0].set_title('Profit Distribution — Before Clipping')
axes[0].set_xlabel('Profit ($)')

axes[1].hist(df['Profit_clipped'], bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
axes[1].set_title('Profit Distribution — After Clipping (1%–99%)')
axes[1].set_xlabel('Profit ($)')

plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation matrix on model features ---
core_num = ['Sales', 'Quantity', 'Discount', 'Order_Month', 'Is_Weekend', 'Profit_clipped']
corr_fe = df_encoded[core_num].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_fe, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, annot_kws={'size': 10})
plt.title('Feature Correlation Matrix (Engineered Dataset)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Train / Test Split ---
from sklearn.model_selection import train_test_split

df_model = df_encoded.dropna(subset=feature_cols + ['Profit_clipped', 'High_Profit']).copy()

X = df_model[feature_cols]
y_reg   = df_model['Profit_clipped']   # continuous target
y_clf   = df_model['High_Profit']       # binary classification target

X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)
_, _, y_train_clf, y_test_clf = train_test_split(
    X, y_clf, test_size=0.2, random_state=42)

print(f"Modelling dataset: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"\nHigh_Profit class distribution (test):\n{y_test_clf.value_counts()}")

# W1: Linear Regression

We begin with Ordinary Least Squares (OLS) linear regression to establish a performance baseline for predicting clipped Profit. OLS minimises the residual sum of squares and provides interpretable feature coefficients, making it a useful first pass before regularisation or dimension-reduction techniques are applied.

**Target:** `Profit_clipped` (continuous)  
**Metrics:** R² and RMSE

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train_reg)
y_pred_lr = lr.predict(X_test)

r2_lr   = r2_score(y_test_reg, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test_reg, y_pred_lr))

print(f"Linear Regression | R²: {r2_lr:.4f} | RMSE: ${rmse_lr:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test_reg, y_pred_lr, alpha=0.4, color='steelblue',
                edgecolors='white', linewidth=0.3)
axes[0].plot([y_test_reg.min(), y_test_reg.max()],
             [y_test_reg.min(), y_test_reg.max()], 'r--', linewidth=2, label='Perfect Fit')
axes[0].set_xlabel('Actual Profit ($)')
axes[0].set_ylabel('Predicted Profit ($)')
axes[0].set_title('Linear Regression — Actual vs. Predicted Profit')
axes[0].legend()
axes[0].annotate(f'R² = {r2_lr:.3f}', xy=(0.05, 0.92), xycoords='axes fraction',
                 fontsize=11, color='darkred')

# Feature coefficients (top 15 by absolute value)
coef_df = pd.Series(lr.coef_, index=feature_cols).reindex(
    pd.Series(lr.coef_, index=feature_cols).abs().nlargest(15).index)
axes[1].barh(coef_df.index, coef_df.values,
             color=['tomato' if v < 0 else 'steelblue' for v in coef_df.values])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Linear Regression — Top 15 Feature Coefficients')
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.show()

# W2: Regularised Regression — Lasso, Ridge, and Elastic Net

Linear regression can overfit when many features are present. Regularisation introduces a penalty on coefficient magnitude to improve generalisation. We tune the penalty strength `alpha` (λ) via GridSearchCV.

| Model | Penalty | Effect |
|---|---|---|
| **Lasso** (L1) | Sum of |coef| | Drives some coefficients to zero (feature selection) |
| **Ridge** (L2) | Sum of coef² | Shrinks all coefficients proportionally |
| **Elastic Net** | Mix of L1 + L2 | Balances sparsity and shrinkage |

In [ ]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.model_selection import GridSearchCV

alphas = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

# --- Lasso ---
lasso_cv = GridSearchCV(Lasso(max_iter=10000), {'alpha': alphas},
                        cv=5, scoring='r2', n_jobs=-1)
lasso_cv.fit(X_train, y_train_reg)
lasso_best = lasso_cv.best_estimator_
print(f"Lasso best alpha: {lasso_cv.best_params_['alpha']}")

# --- Ridge ---
ridge_cv = GridSearchCV(Ridge(), {'alpha': alphas},
                        cv=5, scoring='r2', n_jobs=-1)
ridge_cv.fit(X_train, y_train_reg)
ridge_best = ridge_cv.best_estimator_
print(f"Ridge best alpha: {ridge_cv.best_params_['alpha']}")

# --- Elastic Net ---
en_params = {'alpha': alphas, 'l1_ratio': [0.2, 0.5, 0.8]}
en_cv = GridSearchCV(ElasticNet(max_iter=10000), en_params,
                     cv=5, scoring='r2', n_jobs=-1)
en_cv.fit(X_train, y_train_reg)
en_best = en_cv.best_estimator_
print(f"ElasticNet best params: {en_cv.best_params_}")

In [ ]:
# --- Comparison table ---
results_w2 = {}
for name, m in [('Lasso', lasso_best), ('Ridge', ridge_best), ('ElasticNet', en_best)]:
    preds = m.predict(X_test)
    results_w2[name] = {
        'R²':  round(r2_score(y_test_reg, preds), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test_reg, preds)), 2),
        'MAE':  round(mean_absolute_error(y_test_reg, preds), 2)
    }
    print(f"{name:12s} | R²: {results_w2[name]['R²']:.4f} "
          f"| RMSE: ${results_w2[name]['RMSE']:.2f} "
          f"| MAE: ${results_w2[name]['MAE']:.2f}")

In [ ]:
# --- Lasso coefficient path (features driven to zero) ---
lasso_coefs = pd.Series(lasso_best.coef_, index=feature_cols)
n_zero = (lasso_coefs == 0).sum()
print(f"Lasso zeroed out {n_zero} of {len(feature_cols)} features")

# Plot non-zero coefficients
nonzero = lasso_coefs[lasso_coefs != 0].sort_values()
plt.figure(figsize=(10, max(4, len(nonzero) * 0.3)))
plt.barh(nonzero.index, nonzero.values,
         color=['tomato' if v < 0 else 'steelblue' for v in nonzero.values])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Lasso — Non-Zero Feature Coefficients')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

# W3: Feature Selection, PCR, and PLSR

## Forward / Backward Stepwise Selection

Stepwise selection iteratively adds (forward) or removes (backward) features based on a model performance criterion, helping to identify a parsimonious feature subset. We use a wrapper approach with cross-validated R² as the criterion.

## Principal Components Regression (PCR)

PCR applies PCA to the feature matrix, reducing correlated predictors to orthogonal components, then fits OLS on the retained components. This addresses multicollinearity.

## Partial Least Squares Regression (PLSR)

PLSR similarly decomposes X into latent components but does so while maximising covariance with the target y — making it more directly predictive than PCR when the target is correlated with only a subset of the variance in X.

In [ ]:
# --- Standardise features (required for PCR and PLSR) ---
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [ ]:
# --- Forward Stepwise Selection (greedy wrapper) ---
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

def forward_selection(X_tr, y_tr, feature_names, max_features=15):
    remaining = list(feature_names)
    selected  = []
    best_score = -np.inf

    for _ in range(min(max_features, len(remaining))):
        scores = {}
        for feat in remaining:
            candidate = selected + [feat]
            idx = [feature_names.index(f) for f in candidate]
            cv_score = cross_val_score(
                LinearRegression(), X_tr[:, idx], y_tr,
                cv=5, scoring='r2'
            ).mean()
            scores[feat] = cv_score
        best_feat = max(scores, key=scores.get)
        if scores[best_feat] > best_score:
            selected.append(best_feat)
            remaining.remove(best_feat)
            best_score = scores[best_feat]
        else:
            break
    return selected, best_score

fwd_features, fwd_cv_r2 = forward_selection(
    X_train_sc, y_train_reg.values, feature_cols, max_features=15)
print(f"Forward selection chose {len(fwd_features)} features (CV R² = {fwd_cv_r2:.4f}):")
for f in fwd_features:
    print(f'  {f}')

In [ ]:
# Fit OLS on forward-selected features and evaluate on test set
fwd_idx = [feature_cols.index(f) for f in fwd_features]
lr_fwd  = LinearRegression().fit(X_train_sc[:, fwd_idx], y_train_reg)
y_pred_fwd = lr_fwd.predict(X_test_sc[:, fwd_idx])

r2_fwd   = r2_score(y_test_reg, y_pred_fwd)
rmse_fwd = np.sqrt(mean_squared_error(y_test_reg, y_pred_fwd))
print(f"Forward Selection OLS | R²: {r2_fwd:.4f} | RMSE: ${rmse_fwd:.2f}")

In [ ]:
# --- Principal Components Regression (PCR) ---
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# Choose number of components via CV
best_r2, best_n = -np.inf, 1
for n in range(1, min(21, X_train_sc.shape[1] + 1)):
    pipe = Pipeline([('pca', PCA(n_components=n)), ('ols', LinearRegression())])
    cv = cross_val_score(pipe, X_train_sc, y_train_reg, cv=5, scoring='r2').mean()
    if cv > best_r2:
        best_r2, best_n = cv, n

print(f"PCR best n_components: {best_n} (CV R² = {best_r2:.4f})")

pcr = Pipeline([('pca', PCA(n_components=best_n)), ('ols', LinearRegression())])
pcr.fit(X_train_sc, y_train_reg)
y_pred_pcr = pcr.predict(X_test_sc)

r2_pcr   = r2_score(y_test_reg, y_pred_pcr)
rmse_pcr = np.sqrt(mean_squared_error(y_test_reg, y_pred_pcr))
print(f"PCR ({best_n} components) | R²: {r2_pcr:.4f} | RMSE: ${rmse_pcr:.2f}")

In [ ]:
# --- PCA Explained Variance Plot ---
pca_full = PCA().fit(X_train_sc)
explained = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(explained) + 1), explained, marker='o', markersize=3, color='steelblue')
plt.axvline(best_n, color='red', linestyle='--', label=f'Selected: {best_n} components')
plt.axhline(0.90, color='gray', linestyle=':', label='90% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCR — Cumulative Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Partial Least Squares Regression (PLSR) ---
from sklearn.cross_decomposition import PLSRegression

best_r2_pls, best_n_pls = -np.inf, 1
for n in range(1, 16):
    pls_cv = PLSRegression(n_components=n)
    cv = cross_val_score(pls_cv, X_train_sc, y_train_reg, cv=5, scoring='r2').mean()
    if cv > best_r2_pls:
        best_r2_pls, best_n_pls = cv, n

print(f"PLSR best n_components: {best_n_pls} (CV R² = {best_r2_pls:.4f})")

plsr = PLSRegression(n_components=best_n_pls)
plsr.fit(X_train_sc, y_train_reg)
y_pred_plsr = plsr.predict(X_test_sc).flatten()

r2_plsr   = r2_score(y_test_reg, y_pred_plsr)
rmse_plsr = np.sqrt(mean_squared_error(y_test_reg, y_pred_plsr))
print(f"PLSR ({best_n_pls} components) | R²: {r2_plsr:.4f} | RMSE: ${rmse_plsr:.2f}")

# W4: Logistic Regression with Feature Scaling

Logistic Regression models the probability that an observation belongs to a binary class. Here we classify each order as **High Profit (1)** or **Low Profit (0)** based on whether its clipped profit exceeds the median. Feature scaling is critical for logistic regression — without it, features with larger numeric ranges can dominate the gradient.

We tune the regularisation parameter `C` (inverse of λ) via GridSearchCV and evaluate using accuracy, precision, recall, F1-score, and a confusion matrix.

In [ ]:
from sklearn.linear_model import LogisticRegression

# GridSearchCV over regularisation strength
log_params = {'C': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]}
log_cv = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    log_params, cv=5, scoring='f1', n_jobs=-1
)
log_cv.fit(X_train_sc, y_train_clf)
log_best = log_cv.best_estimator_
print(f"Best C: {log_cv.best_params_['C']}")

y_pred_log = log_best.predict(X_test_sc)
print(f"\nAccuracy: {accuracy_score(y_test_clf, y_pred_log):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_clf, y_pred_log, target_names=['Low Profit', 'High Profit']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_clf, y_pred_log)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low Profit', 'High Profit'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Top feature coefficients from Logistic Regression
log_coef = pd.Series(log_best.coef_[0], index=feature_cols)
top_log = log_coef.abs().nlargest(15).index
log_coef_top = log_coef[top_log].sort_values()

plt.figure(figsize=(10, 5))
plt.barh(log_coef_top.index, log_coef_top.values,
         color=['tomato' if v < 0 else 'steelblue' for v in log_coef_top.values])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression — Top 15 Feature Coefficients')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

# W5: Support Vector Machines (SVM)

Support Vector Machines find the hyperplane that maximises the margin between classes. The **kernel trick** allows SVMs to operate in high-dimensional feature spaces without explicitly computing transformations, enabling non-linear decision boundaries.

| Kernel | Use case |
|---|---|
| Linear | Linearly separable data |
| RBF (Gaussian) | Non-linear boundaries — most common default |
| Polynomial | Curved boundaries |

**Regularisation parameter C:** Higher C = less tolerance for misclassification (tighter margin, risk of overfitting). Lower C = wider margin, more misclassifications allowed.

We apply SVM to the binary classification task (`High_Profit`). Because SVM is sensitive to scale, we use the already-standardised `X_train_sc`.

In [ ]:
from sklearn.svm import SVC

# GridSearch over kernel and C
svm_params = {
    'C':      [0.1, 1.0, 10.0],
    'kernel': ['linear', 'rbf'],
    'gamma':  ['scale']
}
svm_cv = GridSearchCV(
    SVC(random_state=42, probability=True),
    svm_params, cv=5, scoring='f1', n_jobs=-1
)
svm_cv.fit(X_train_sc, y_train_clf)
svm_best = svm_cv.best_estimator_
print(f"Best SVM params: {svm_cv.best_params_}")

y_pred_svm = svm_best.predict(X_test_sc)
print(f"\nAccuracy: {accuracy_score(y_test_clf, y_pred_svm):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_clf, y_pred_svm, target_names=['Low Profit', 'High Profit']))

In [ ]:
# SVM Confusion Matrix
cm_svm = confusion_matrix(y_test_clf, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm,
                                  display_labels=['Low Profit', 'High Profit'])

fig, ax = plt.subplots(figsize=(6, 5))
disp_svm.plot(ax=ax, colorbar=False, cmap='Purples')
ax.set_title(f"SVM ({svm_best.kernel} kernel) — Confusion Matrix")
plt.tight_layout()
plt.show()

# W6: Decision Tree and Random Forest

## Decision Tree
Decision Trees recursively partition the feature space based on the feature and threshold that maximise information gain (or minimise impurity). They are highly interpretable but prone to overfitting without depth constraints.

## Random Forest
Random Forest is an ensemble of decision trees trained on bootstrap samples of the data, each using a random subset of features at each split. Averaging across trees reduces variance and produces a far more robust model. In e-commerce, Random Forests can power recommendation engines and demand forecasting.

We apply both to the **binary classification** task and compare results.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# --- Decision Tree with GridSearchCV ---
dt_params = {'max_depth': [3, 5, 10, None], 'min_samples_leaf': [1, 5, 10]}
dt_cv = GridSearchCV(
    DecisionTreeClassifier(random_state=42), dt_params,
    cv=5, scoring='f1', n_jobs=-1
)
dt_cv.fit(X_train, y_train_clf)  # trees don't require scaling
dt_best = dt_cv.best_estimator_
print(f"Best DT params: {dt_cv.best_params_}")

y_pred_dt = dt_best.predict(X_test)
print(f"\nDecision Tree — Accuracy: {accuracy_score(y_test_clf, y_pred_dt):.4f}")
print(classification_report(y_test_clf, y_pred_dt, target_names=['Low Profit', 'High Profit']))

In [ ]:
# Visualise the pruned tree (max depth 3 for readability)
dt_vis = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_vis.fit(X_train, y_train_clf)

plt.figure(figsize=(20, 7))
plot_tree(dt_vis, feature_names=feature_cols,
          class_names=['Low', 'High'], filled=True,
          rounded=True, fontsize=8, max_depth=3)
plt.title('Decision Tree (depth=3 for visualisation)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Random Forest with GridSearchCV ---
rf_params = {
    'n_estimators':   [100, 200],
    'max_depth':      [5, 10, 15, None],
    'min_samples_leaf': [1, 5]
}
rf_cv = GridSearchCV(
    RandomForestClassifier(random_state=42), rf_params,
    cv=5, scoring='f1', n_jobs=-1
)
rf_cv.fit(X_train, y_train_clf)
rf_best = rf_cv.best_estimator_
print(f"Best RF params: {rf_cv.best_params_}")

y_pred_rf = rf_best.predict(X_test)
print(f"\nRandom Forest — Accuracy: {accuracy_score(y_test_clf, y_pred_rf):.4f}")
print(classification_report(y_test_clf, y_pred_rf, target_names=['Low Profit', 'High Profit']))

In [ ]:
# Random Forest Feature Importance
fi = pd.Series(rf_best.feature_importances_, index=feature_cols)
fi_top = fi.nlargest(15)

plt.figure(figsize=(10, 5))
fi_top.sort_values().plot(kind='barh', color='steelblue')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Random Forest — Top 15 Feature Importances')
plt.xlabel('Importance (Mean Decrease in Impurity)')
plt.tight_layout()
plt.show()

In [ ]:
# Random Forest Confusion Matrix
cm_rf = confusion_matrix(y_test_clf, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf,
                                 display_labels=['Low Profit', 'High Profit'])

fig, ax = plt.subplots(figsize=(6, 5))
disp_rf.plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.show()

# Bonus: K-Means Clustering — Regional Market Segmentation

As proposed in the capstone proposal, K-Means Clustering is used to segment cities/regions based on purchasing behaviour, profitability, and product category mix. This unsupervised approach reveals natural groupings without predefined labels, identifying which markets share similar profiles and which are underperforming.

**Features used:** Total Sales, Total Profit, Average Discount, Total Quantity (aggregated by city)

In [ ]:
from sklearn.cluster import KMeans

# Aggregate to city level
city_agg = df.groupby('City').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', 'mean'),
    Total_Quantity=('Quantity', 'sum'),
    Num_Orders=('Order ID', 'nunique')
).reset_index()

# Scale for KMeans
from sklearn.preprocessing import StandardScaler
cluster_features = ['Total_Sales', 'Total_Profit', 'Avg_Discount', 'Total_Quantity']
scaler_k = StandardScaler()
X_cluster = scaler_k.fit_transform(city_agg[cluster_features])

# Elbow method to choose k
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, marker='o', color='steelblue')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Within-Cluster SSE)')
plt.title('K-Means Elbow Plot — City-Level Market Segmentation')
plt.tight_layout()
plt.show()

In [ ]:
# Fit K-Means with chosen k (adjust based on elbow plot)
k_chosen = 4
km_final = KMeans(n_clusters=k_chosen, random_state=42, n_init=10)
city_agg['Cluster'] = km_final.fit_predict(X_cluster)

# Cluster summary
cluster_summary = city_agg.groupby('Cluster')[cluster_features + ['Num_Orders']].mean().round(2)
display(cluster_summary)

# Scatter: Total Sales vs Total Profit, coloured by cluster
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
plt.figure(figsize=(10, 6))
for c in range(k_chosen):
    subset = city_agg[city_agg['Cluster'] == c]
    plt.scatter(subset['Total_Sales'], subset['Total_Profit'],
                label=f'Cluster {c}', alpha=0.7, color=palette[c], s=60)
plt.xlabel('Total Sales ($)')
plt.ylabel('Total Profit ($)')
plt.title(f'K-Means Clustering (k={k_chosen}) — Cities by Sales & Profit')
plt.legend()
plt.axhline(0, color='black', linewidth=0.6, linestyle='--')
plt.tight_layout()
plt.show()

# Model Comparison Summary

The table below consolidates performance metrics across all models trained in Weeks 1–6. Regression models are evaluated on R² and RMSE against `Profit_clipped`. Classification models are evaluated on Accuracy and F1-Score for `High_Profit`.

In [ ]:
# --- Regression models (W1–W3) ---
reg_results = {
    'Linear Regression':  {'R²': r2_lr,   'RMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_lr))},
    'Lasso':              {'R²': results_w2['Lasso']['R²'],      'RMSE': results_w2['Lasso']['RMSE']},
    'Ridge':              {'R²': results_w2['Ridge']['R²'],      'RMSE': results_w2['Ridge']['RMSE']},
    'ElasticNet':         {'R²': results_w2['ElasticNet']['R²'], 'RMSE': results_w2['ElasticNet']['RMSE']},
    'Forward Sel. OLS':   {'R²': r2_fwd,   'RMSE': rmse_fwd},
    'PCR':                {'R²': r2_pcr,   'RMSE': rmse_pcr},
    'PLSR':               {'R²': r2_plsr,  'RMSE': rmse_plsr},
}
reg_df = pd.DataFrame(reg_results).T.astype(float).round(4)
print("--- Regression Model Comparison (target: Profit_clipped) ---")
display(reg_df.sort_values('R²', ascending=False))

In [ ]:
# --- Classification models (W4–W6) ---
from sklearn.metrics import f1_score

clf_results = {
    'Logistic Regression': {
        'Accuracy': accuracy_score(y_test_clf, y_pred_log),
        'F1-Score': f1_score(y_test_clf, y_pred_log)
    },
    'SVM': {
        'Accuracy': accuracy_score(y_test_clf, y_pred_svm),
        'F1-Score': f1_score(y_test_clf, y_pred_svm)
    },
    'Decision Tree': {
        'Accuracy': accuracy_score(y_test_clf, y_pred_dt),
        'F1-Score': f1_score(y_test_clf, y_pred_dt)
    },
    'Random Forest': {
        'Accuracy': accuracy_score(y_test_clf, y_pred_rf),
        'F1-Score': f1_score(y_test_clf, y_pred_rf)
    },
}
clf_df = pd.DataFrame(clf_results).T.astype(float).round(4)
print("--- Classification Model Comparison (target: High_Profit) ---")
display(clf_df.sort_values('F1-Score', ascending=False))